# 🩹 HANDLING MISSING VALUES (EKSİK DEĞER YÖNETİMİ)
> **🎯 AMAÇ:** Veri setindeki NaN değerlerini tespit ve doldurma  
> **📥 GİRDİ:** Ham DataFrame  
> **📤 ÇIKTI:** Temiz DataFrame, eksik değer raporu  
### 🔧 YÖNTEM SEÇİMİ
| Eksiklik | Öneri |
|----------|-------|
| >%60 | Sütunu sil |
| %20-60 | KNN Imputer |
| <%20 Sayısal | Median |
| <%20 Kategorik | Mod |
### ⚠️ UYARI
Imputer'i SADECE X_train'de fit et!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
print('=' * 50 + '\n🩹 EKSİK DEĞER YÖNETİMİ BAŞLATILIYOR\n' + '=' * 50)
# df = pd.read_csv('veri.csv')  # <--- DAYI BURAYI DOLDUR (!!!)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing/len(df)*100).round(2)
missing_df = pd.DataFrame({'Eksik': missing, 'Eksik%': missing_pct})
missing_df = missing_df[missing_df['Eksik']>0].sort_values('Eksik%', ascending=False)
print(f'Toplam satir: {len(df)} | Eksik sutun: {len(missing_df)} | Eksik hucre: {df.isnull().sum().sum()}')
print(missing_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if len(missing_df) > 0:
    colors = ['#FF4444' if p>40 else '#FF8C00' if p>20 else '#2196F3' for p in missing_df['Eksik%']]
    missing_df['Eksik%'].plot(kind='barh', ax=axes[0], color=colors)
    axes[0].axvline(60, color='red', linestyle='--', alpha=0.6, label='Sil siniri')
    axes[0].axvline(20, color='orange', linestyle='--', alpha=0.6, label='KNN siniri')
    axes[0].legend(); axes[0].set_title('Eksik Deger Orani')
sns.heatmap(df.iloc[:50].isnull(), cbar=False, cmap='Reds', ax=axes[1], yticklabels=False)
axes[1].set_title('Eksik Deger Haritasi (Ilk 50 satir)')
plt.tight_layout(); plt.show()

In [ ]:
# >%60 -> sil
to_drop = missing_df[missing_df['Eksik%']>60].index.tolist()
if to_drop: df.drop(columns=to_drop, inplace=True); print(f'🗑️ Silindi: {to_drop}')

# Sayisal -> median
num_cols = df.select_dtypes(include=[float,int]).columns.tolist()
if num_cols:
    imp = SimpleImputer(strategy='median')
    df[num_cols] = imp.fit_transform(df[num_cols])
    print(f'✅ Sayisal median ile dolduruldu: {num_cols}')

# Kategorik -> mod
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_cols:
    imp_c = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = imp_c.fit_transform(df[cat_cols])
    print(f'✅ Kategorik mod ile dolduruldu: {cat_cols}')

# KNN Imputer:
# knn = KNNImputer(n_neighbors=5)
# X_train_imp = knn.fit_transform(X_train)
# X_test_imp  = knn.transform(X_test)

print(f'\n✅ Kalan eksik hucre: {df.isnull().sum().sum()} | Shape: {df.shape}')